# StaySure Hotel Cancellation ML
## Milestone 3 — Preprocessing and Chronological Splitting

**Business goal:** create a reproducible, leakage-safe preprocessing workflow that converts cleaned hotel reservations into model-ready training, validation, and test matrices.

This notebook is learner-led. Read each explanation, run the completed examples, and finish the `TODO` sections yourself before checking the milestone checklist.

### Main outputs

- `models/preprocessing_pipeline.joblib` — local fitted preprocessing pipeline (ignored by Git)
- `reports/milestone_03_split_metrics.csv` — split sizes, dates, and cancellation rates
- `reports/milestone_03_feature_manifest.csv` — model input features and their roles
- `reports/milestone_03_preprocessing_decisions.csv` — documented preprocessing choices

**Important:** preprocessing must be fitted on the training data only. Validation and test data simulate future, unseen reservations.


### What you should be able to explain afterward

1. Why a chronological split is appropriate for hotel reservations.
2. Why preprocessing is fitted on training data only.
3. How a `Pipeline` and `ColumnTransformer` prevent inconsistent transformations.
4. Why categorical variables require encoding.
5. Why `handle_unknown="ignore"` matters for future categories.
6. When scaling numerical features matters.


## 1. Setup


In [1]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

print("Libraries loaded.")


Libraries loaded.


The next cell finds the project root whether this notebook is launched from the repository root or from the `notebooks/` folder.


In [2]:
cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd

CLEAN_DATA_PATH = PROJECT_ROOT / "data" / "processed" / "hotel_bookings_clean.csv"
REPORTS_DIR = PROJECT_ROOT / "reports"
MODELS_DIR = PROJECT_ROOT / "models"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Clean data path:", CLEAN_DATA_PATH)


Project root: C:\Users\theko\ML_Projects\staysure-hotel-cancellation-ml
Clean data path: C:\Users\theko\ML_Projects\staysure-hotel-cancellation-ml\data\processed\hotel_bookings_clean.csv


## 2. Load and verify the Milestone 2 dataset


In [3]:
if not CLEAN_DATA_PATH.exists():
    raise FileNotFoundError(
        f"Clean dataset not found at {CLEAN_DATA_PATH}. "
        "Run Milestone 2 and save the processed CSV first."
    )

df = pd.read_csv(CLEAN_DATA_PATH)

print("Cleaned shape:", df.shape)
print("Cancellation rate:", f"{df['is_canceled'].mean():.2%}")
display(df.head())


Cleaned shape: (118564, 28)
Cancellation rate: 37.26%


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,country,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,deposit_type,agent,customer_type,adr,required_car_parking_spaces,total_of_special_requests,total_guests,total_nights
0,Resort Hotel,0,7,2015,July,27,1,0,1,1,0.0000,0,BB,GBR,Direct,Direct,0,0,0,A,No Deposit,0.0000,Transient,75.0000,0,0,1.0000,1
1,Resort Hotel,0,13,2015,July,27,1,0,1,1,0.0000,0,BB,GBR,Corporate,Corporate,0,0,0,A,No Deposit,304.0000,Transient,75.0000,0,0,1.0000,1
2,Resort Hotel,0,14,2015,July,27,1,0,2,2,0.0000,0,BB,GBR,Online TA,TA/TO,0,0,0,A,No Deposit,240.0000,Transient,98.0000,0,1,2.0000,2
3,Resort Hotel,0,14,2015,July,27,1,0,2,2,0.0000,0,BB,GBR,Online TA,TA/TO,0,0,0,A,No Deposit,240.0000,Transient,98.0000,0,1,2.0000,2
4,Resort Hotel,0,0,2015,July,27,1,0,2,2,0.0000,0,BB,PRT,Direct,Direct,0,0,0,C,No Deposit,0.0000,Transient,107.0000,0,0,2.0000,2


In [4]:
TARGET = "is_canceled"
FORBIDDEN_LEAKAGE = {
    "reservation_status",
    "reservation_status_date",
    "assigned_room_type",
    "booking_changes",
    "days_in_waiting_list",
}

assert TARGET in df.columns, f"Missing target column: {TARGET}"
assert df[TARGET].isin([0, 1]).all(), "Target must contain only 0 and 1."
assert not (FORBIDDEN_LEAKAGE & set(df.columns)), "Leakage columns are still present."
assert df.isna().sum().sum() == 0, "Milestone 2 data still contains missing values."

print("Data contract passed.")


Data contract passed.


## 3. Create a real arrival date

A chronological split needs an actual date rather than three separate year, month, and day columns. The new `arrival_date` column will control the split, but it will not be sent directly into the preprocessing pipeline.


In [5]:
month_number = {
    "January": 1,
    "February": 2,
    "March": 3,
    "April": 4,
    "May": 5,
    "June": 6,
    "July": 7,
    "August": 8,
    "September": 9,
    "October": 10,
    "November": 11,
    "December": 12,
}

df["arrival_date"] = pd.to_datetime(
    {
        "year": df["arrival_date_year"],
        "month": df["arrival_date_month"].map(month_number),
        "day": df["arrival_date_day_of_month"],
    },
    errors="raise",
)

df = df.sort_values("arrival_date").reset_index(drop=True)

print("Date range:", df["arrival_date"].min().date(), "to", df["arrival_date"].max().date())
display(df[["arrival_date", "arrival_date_year", "arrival_date_month", "arrival_date_day_of_month"]].head())


Date range: 2015-07-01 to 2017-08-31


,arrival_date,arrival_date_year,arrival_date_month,arrival_date_day_of_month
0,2015-07-01,2015,July,1
1,2015-07-01,2015,July,1
2,2015-07-01,2015,July,1
3,2015-07-01,2015,July,1
4,2015-07-01,2015,July,1


### TODO 1 — Choose chronological split dates

Recommended cutoffs:

- Training: through **2017-02-10**
- Validation: **2017-02-11** through **2017-05-21**
- Test: **2017-05-22** onward

These boundaries produce approximately a 70% / 15% / 15% split and keep each calendar day in only one dataset.


In [6]:
# TODO 1: Replace None with the recommended dates above in YYYY-MM-DD format.
TRAIN_END_DATE = "2017-02-10"
VALID_END_DATE = "2017-05-21"

if TRAIN_END_DATE is None or VALID_END_DATE is None:
    print("TODO 1: Enter both split dates before continuing.")
    train_df = valid_df = test_df = None
else:
    train_end = pd.Timestamp(TRAIN_END_DATE)
    valid_end = pd.Timestamp(VALID_END_DATE)

    assert train_end < valid_end, "Training must end before validation ends."

    train_df = df[df["arrival_date"] <= train_end].copy()
    valid_df = df[(df["arrival_date"] > train_end) & (df["arrival_date"] <= valid_end)].copy()
    test_df = df[df["arrival_date"] > valid_end].copy()

    assert len(train_df) + len(valid_df) + len(test_df) == len(df)
    assert min(len(train_df), len(valid_df), len(test_df)) > 0
    assert train_df["arrival_date"].max() < valid_df["arrival_date"].min()
    assert valid_df["arrival_date"].max() < test_df["arrival_date"].min()

    print("Chronological split created.")


Chronological split created.


In [7]:
if train_df is None:
    print("Complete TODO 1 first.")
else:
    split_preview = pd.DataFrame(
        {
            "split": ["train", "validation", "test"],
            "rows": [len(train_df), len(valid_df), len(test_df)],
            "share": [len(train_df) / len(df), len(valid_df) / len(df), len(test_df) / len(df)],
            "start_date": [
                train_df["arrival_date"].min(),
                valid_df["arrival_date"].min(),
                test_df["arrival_date"].min(),
            ],
            "end_date": [
                train_df["arrival_date"].max(),
                valid_df["arrival_date"].max(),
                test_df["arrival_date"].max(),
            ],
            "cancellation_rate": [
                train_df[TARGET].mean(),
                valid_df[TARGET].mean(),
                test_df[TARGET].mean(),
            ],
        }
    )
    display(split_preview)


,split,rows,share,start_date,end_date,cancellation_rate
0,train,82937,0.6995,2015-07-01,2017-02-10,0.3630
1,validation,17758,0.1498,2017-02-11,2017-05-21,0.3947
2,test,17869,0.1507,2017-05-22,2017-08-31,0.3954


**TODO 1 explanation:** In your own words, explain why randomly mixing older and newer reservations could give an unrealistic estimate of future performance.

## Data Split Explanation
Randomly mixing older and newer reservations would allow the training and evaluation sets to contain information from the same time periods. This could produce an overly optimistic performance estimate because it would not represent how the model will actually be used. A chronological split trains the model on earlier reservations and evaluates it on later reservations, which better measures its ability to predict future booking behavior and exposes changes in cancellation patterns over time.


## 4. Separate predictors from the target


The target is what the model will predict. The features are the information supplied to the model. `arrival_date` controls the split but is excluded from the feature matrix because scikit-learn's basic preprocessing steps do not directly handle timestamps.


In [9]:
if train_df is None:
    print("Complete TODO 1 first.")
    FEATURE_COLUMNS = None
    X_train = X_valid = X_test = None
    y_train = y_valid = y_test = None
else:
    EXCLUDED_FROM_MODEL = [TARGET, "arrival_date"]
    FEATURE_COLUMNS = [column for column in df.columns if column not in EXCLUDED_FROM_MODEL]

    # Completed example:
    X_train = train_df[FEATURE_COLUMNS].copy()
    y_train = train_df[TARGET].copy()

    # TODO 2: Create X_valid, y_valid, X_test, and y_test using the same pattern.
    X_valid = valid_df[FEATURE_COLUMNS].copy()
    y_valid = valid_df[TARGET].copy()
    X_test = test_df[FEATURE_COLUMNS].copy()
    y_test = test_df[TARGET].copy()

    print("Training features:", X_train.shape)
    print("Training target:", y_train.shape)


Training features: (82937, 27)
Training target: (82937,)


### TODO 2 — Confirm the feature/target split

After completing the four lines above, check that:

- every `X` table has the same number of columns;
- each `X` table and matching `y` series have the same number of rows;
- the target does not appear in the feature columns.


In [10]:
if any(item is None for item in [X_train, X_valid, X_test, y_train, y_valid, y_test]):
    print("TODO 2: Create all validation and test feature/target objects.")
else:
    assert X_train.shape[1] == X_valid.shape[1] == X_test.shape[1]
    assert len(X_train) == len(y_train)
    assert len(X_valid) == len(y_valid)
    assert len(X_test) == len(y_test)
    assert TARGET not in X_train.columns
    assert not (FORBIDDEN_LEAKAGE & set(X_train.columns))

    print("Feature/target checks passed.")


Feature/target checks passed.


## 5. Identify numerical and categorical features

Text categories need encoding. Numerical columns can be imputed and scaled. `agent` is an identifier, not a quantity, so the recommended choice is to treat it as a categorical feature.


In [11]:
# TODO 3: Replace None with True or False.
TREAT_AGENT_AS_CATEGORY = True

if X_train is None or TREAT_AGENT_AS_CATEGORY is None:
    print("TODO 3: Decide whether agent should be treated as a category.")
    categorical_features = numeric_features = None
else:
    if TREAT_AGENT_AS_CATEGORY:
        for frame in [X_train, X_valid, X_test]:
            frame["agent"] = frame["agent"].astype("Int64").astype(str)

    categorical_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
    numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()

    assert set(categorical_features).isdisjoint(numeric_features)
    assert set(categorical_features + numeric_features) == set(X_train.columns)

    print("Categorical features:", len(categorical_features))
    print(categorical_features)
    print("\nNumerical features:", len(numeric_features))
    print(numeric_features)


Categorical features: 10
['hotel', 'arrival_date_month', 'meal', 'country', 'market_segment', 'distribution_channel', 'reserved_room_type', 'deposit_type', 'agent', 'customer_type']

Numerical features: 17
['lead_time', 'arrival_date_year', 'arrival_date_week_number', 'arrival_date_day_of_month', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'adr', 'required_car_parking_spaces', 'total_of_special_requests', 'total_guests', 'total_nights']


C:\Users\theko\AppData\Local\Temp\ipykernel_12940\4021804651.py:12: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()


**TODO 3 explanation:** Why would treating an agent ID as a continuous number create a misleading relationship?
## AGENT ID as CATEGORY Decision
An agent ID identifies a booking agent but does not measure an amount. Treating it as a continuous number would incorrectly suggest that higher agent numbers are larger or have an ordered relationship with lower agent numbers. Treating agent as categorical allows the model to distinguish agents without assuming that the numerical distance between their IDs has meaning.


## 6. Build a reusable preprocessing pipeline


The numerical pipeline fills missing values with the training median and scales each numeric feature. The categorical pipeline fills missing values with the training mode and one-hot encodes categories.

Milestone 2 currently produced a dataset with no missing values, but including imputers makes the production pipeline safer if future data contains missing entries.


In [12]:
if categorical_features is None or numeric_features is None:
    print("Complete TODO 3 first.")
    preprocessor = None
else:
    numeric_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            # TODO 4: Add handle_unknown="ignore" inside OneHotEncoder(...).
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("numeric", numeric_pipeline, numeric_features),
            ("categorical", categorical_pipeline, categorical_features),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )

    print(preprocessor)


ColumnTransformer(transformers=[('numeric',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['lead_time', 'arrival_date_year',
                                  'arrival_date_week_number',
                                  'arrival_date_day_of_month',
                                  'stays_in_weekend_nights',
                                  'stays_in_week_nights', 'adults', 'children',
                                  'babies', 'is_repeated_guest',
                                  'previous_cancellations',
                                  'prev...
                                  'total_of_special_requests', 'total_guests',
                                  'total_nights']),
                                ('categorical',
                                 Pipeline(steps=

### TODO 4 — Protect the pipeline from unseen categories

Add `handle_unknown="ignore"` to `OneHotEncoder`. Without it, a category that appears in validation, test, or production—but not in training—can cause the transformation to fail.

**Explain the decision:** Why should the validation and test sets not teach the encoder which categories exist?

## ENCODER Decision 
The encoder should learn its category vocabulary from the training data only. Allowing validation or test data to teach the encoder which categories exist would allow information from future reservations to influence preprocessing, creating data leakage. Setting handle_unknown="ignore" allows the pipeline to process previously unseen categories without refitting the encoder or using future information.


## 7. Fit on training data; transform validation and test data


In [13]:
if preprocessor is None or X_valid is None or X_test is None:
    print("Complete TODOs 1–4 first.")
    X_train_ready = X_valid_ready = X_test_ready = None
else:
    # Completed example: learn preprocessing rules from training data.
    X_train_ready = preprocessor.fit_transform(X_train)

    # TODO 5: Use transform (not fit_transform) on validation and test data.
    X_valid_ready = preprocessor.transform(X_valid)
    X_test_ready = preprocessor.transform(X_test)

    print("Training matrix shape:", X_train_ready.shape)


Training matrix shape: (82937, 522)


### TODO 5 — Prevent preprocessing leakage

Complete the two transformation lines above. Then explain why fitting the scaler, imputer, or encoder on validation/test data would leak information from the future.

## TRANSFORM DATA VALIDATION Decision
Fitting the scaler, imputer, or encoder on validation or test data would allow information from future reservations to influence the preprocessing rules. I fit the preprocessing pipeline only on the training data using fit_transform(). I then use transform() on the validation and test data so that they remain unseen evaluation datasets. This prevents data leakage and provides a more realistic estimate of how the model will perform on future bookings.


In [14]:
if any(item is None for item in [X_train_ready, X_valid_ready, X_test_ready]):
    print("TODO 5: Transform validation and test features.")
    transformed_feature_names = None
else:
    assert X_train_ready.shape[1] == X_valid_ready.shape[1] == X_test_ready.shape[1]
    assert X_train_ready.shape[0] == len(X_train)
    assert X_valid_ready.shape[0] == len(X_valid)
    assert X_test_ready.shape[0] == len(X_test)

    transformed_feature_names = preprocessor.get_feature_names_out()

    print("Model-ready shapes")
    print("Train:", X_train_ready.shape)
    print("Validation:", X_valid_ready.shape)
    print("Test:", X_test_ready.shape)
    print("Transformed features:", len(transformed_feature_names))
    print("First 20 names:", transformed_feature_names[:20].tolist())


Model-ready shapes
Train: (82937, 522)
Validation: (17758, 522)
Test: (17869, 522)
Transformed features: 522
First 20 names: ['lead_time', 'arrival_date_year', 'arrival_date_week_number', 'arrival_date_day_of_month', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'adr', 'required_car_parking_spaces', 'total_of_special_requests', 'total_guests', 'total_nights', 'hotel_City Hotel', 'hotel_Resort Hotel', 'arrival_date_month_April']


## 8. Document the split and preprocessing decisions


In [15]:
if transformed_feature_names is None:
    print("Complete TODOs 1–5 first.")
    split_metrics = feature_manifest = preprocessing_decisions = None
else:
    split_metrics = pd.DataFrame(
        {
            "split": ["train", "validation", "test"],
            "rows": [len(train_df), len(valid_df), len(test_df)],
            "columns_before_preprocessing": [X_train.shape[1], X_valid.shape[1], X_test.shape[1]],
            "columns_after_preprocessing": [
                X_train_ready.shape[1],
                X_valid_ready.shape[1],
                X_test_ready.shape[1],
            ],
            "start_date": [
                train_df["arrival_date"].min().date(),
                valid_df["arrival_date"].min().date(),
                test_df["arrival_date"].min().date(),
            ],
            "end_date": [
                train_df["arrival_date"].max().date(),
                valid_df["arrival_date"].max().date(),
                test_df["arrival_date"].max().date(),
            ],
            "cancellation_rate": [y_train.mean(), y_valid.mean(), y_test.mean()],
        }
    )

    feature_manifest = pd.DataFrame(
        {
            "source_feature": FEATURE_COLUMNS,
            "preprocessing_role": [
                "categorical" if column in categorical_features else "numeric"
                for column in FEATURE_COLUMNS
            ],
        }
    )

    preprocessing_decisions = pd.DataFrame(
        {
            "decision": [
                "split_strategy",
                "training_end_date",
                "validation_end_date",
                "agent_treatment",
                "numeric_missing_values",
                "numeric_scaling",
                "categorical_missing_values",
                "categorical_encoding",
                "unknown_categories",
                "pipeline_fit_scope",
            ],
            "choice": [
                "chronological",
                str(pd.Timestamp(TRAIN_END_DATE).date()),
                str(pd.Timestamp(VALID_END_DATE).date()),
                "categorical" if TREAT_AGENT_AS_CATEGORY else "numeric",
                "median imputation",
                "StandardScaler",
                "most-frequent imputation",
                "one-hot encoding",
                "ignore during transform",
                "training data only",
            ],
        }
    )

    display(split_metrics)
    display(feature_manifest)
    display(preprocessing_decisions)


,split,rows,columns_before_preprocessing,columns_after_preprocessing,start_date,end_date,cancellation_rate
0,train,82937,27,522,2015-07-01,2017-02-10,0.3630
1,validation,17758,27,522,2017-02-11,2017-05-21,0.3947
2,test,17869,27,522,2017-05-22,2017-08-31,0.3954


,source_feature,preprocessing_role
0,hotel,categorical
1,lead_time,numeric
2,arrival_date_year,numeric
3,arrival_date_month,categorical
4,arrival_date_week_number,numeric
5,arrival_date_day_of_month,numeric
6,stays_in_weekend_nights,numeric
7,stays_in_week_nights,numeric
8,adults,numeric
9,children,numeric


,decision,choice
0,split_strategy,chronological
1,training_end_date,2017-02-10
2,validation_end_date,2017-05-21
3,agent_treatment,categorical
4,numeric_missing_values,median imputation
5,numeric_scaling,StandardScaler
6,categorical_missing_values,most-frequent imputation
7,categorical_encoding,one-hot encoding
8,unknown_categories,ignore during transform
9,pipeline_fit_scope,training data only


### TODO 6 — Interpret the split

Compare the cancellation rates across training, validation, and test data.

1. Did the cancellation rate change over time?
2. Why is this useful to discover before modeling?
3. What could happen if a model is trained only on older behavior?

## CHRONOLOGICAL SPLIT EXPLAINED
The cancellation rate increased from 36.30% in the training period to 39.47% in validation and 39.54% in testing. This suggests that cancellation behavior changed over time. Identifying this shift before modeling is important because a model trained on older reservations may underpredict cancellations in newer periods. Using a chronological split therefore provides a more realistic evaluation of how the model may perform on future bookings.


## 9. Save the pipeline and reports


Review every TODO and validation check first. The fitted `.joblib` pipeline remains local because model artifacts are ignored by Git. The small CSV reports should be committed.


In [16]:
# TODO 7: Change to True only after completing and reviewing TODOs 1–6.
SAVE_OUTPUTS = True

PIPELINE_PATH = MODELS_DIR / "preprocessing_pipeline.joblib"
SPLIT_REPORT_PATH = REPORTS_DIR / "milestone_03_split_metrics.csv"
FEATURE_REPORT_PATH = REPORTS_DIR / "milestone_03_feature_manifest.csv"
DECISIONS_REPORT_PATH = REPORTS_DIR / "milestone_03_preprocessing_decisions.csv"

if not SAVE_OUTPUTS:
    print("Outputs not saved. Complete the notebook, then set SAVE_OUTPUTS = True.")
elif any(item is None for item in [
    preprocessor,
    split_metrics,
    feature_manifest,
    preprocessing_decisions,
]):
    raise RuntimeError("Complete all TODOs and validation checks before saving.")
else:
    joblib.dump(preprocessor, PIPELINE_PATH)
    split_metrics.to_csv(SPLIT_REPORT_PATH, index=False)
    feature_manifest.to_csv(FEATURE_REPORT_PATH, index=False)
    preprocessing_decisions.to_csv(DECISIONS_REPORT_PATH, index=False)

    print("Saved local pipeline:", PIPELINE_PATH)
    print("Saved report:", SPLIT_REPORT_PATH)
    print("Saved report:", FEATURE_REPORT_PATH)
    print("Saved report:", DECISIONS_REPORT_PATH)


Saved local pipeline: C:\Users\theko\ML_Projects\staysure-hotel-cancellation-ml\models\preprocessing_pipeline.joblib
Saved report: C:\Users\theko\ML_Projects\staysure-hotel-cancellation-ml\reports\milestone_03_split_metrics.csv
Saved report: C:\Users\theko\ML_Projects\staysure-hotel-cancellation-ml\reports\milestone_03_feature_manifest.csv
Saved report: C:\Users\theko\ML_Projects\staysure-hotel-cancellation-ml\reports\milestone_03_preprocessing_decisions.csv


In [17]:
gitignore_path = PROJECT_ROOT / ".gitignore"
gitignore_text = gitignore_path.read_text(encoding="utf-8") if gitignore_path.exists() else ""

processed_rule_found = "data/processed/*.csv" in gitignore_text
model_rule_found = "models/*.joblib" in gitignore_text

print("Processed-data ignore rule found." if processed_rule_found else "Missing: data/processed/*.csv")
print("Model ignore rule found." if model_rule_found else "Missing: models/*.joblib")


Processed-data ignore rule found.
Model ignore rule found.


## Interview-ready explanation

Use this structure, but explain it in your own words:

> I converted the reservation fields into a true arrival date and split the data chronologically, so validation and test sets represented future bookings. I separated the target from the predictors and used a scikit-learn `ColumnTransformer` to apply median imputation and scaling to numerical features and most-frequent imputation plus one-hot encoding to categorical features. I fitted the preprocessing pipeline only on training data, then reused it to transform validation and test data. This prevented future information from influencing the learned preprocessing rules and produced consistent model-ready matrices.


## Questions you should be ready to answer

1. Why did you use a chronological split instead of a random split?
2. Why is `fit_transform` used for training but only `transform` for validation and test data?
3. What does one-hot encoding accomplish?
4. Why did you use `handle_unknown="ignore"`?
5. Why did you treat `agent` as a category rather than a continuous measurement?
6. Does a tree-based model require scaling? Why keep scaling in a reusable pipeline?
7. How does a scikit-learn pipeline reduce leakage and deployment errors?

## PREPROCESSING Interview-ready Answers
1. Why did you use a chronological split?
I used a chronological split because the model will be trained on historical reservations and used to predict future cancellations. A random split could mix older and newer bookings across the datasets, creating an overly optimistic evaluation. The chronological split more realistically tests whether the model generalizes to future booking behavior.
2. Why use fit_transform() for training but transform() for validation and testing?
fit_transform() allows the preprocessing pipeline to learn rules—such as medians, means, scaling values, and category vocabularies—from the training data and then apply them. Validation and test data use only transform() so they cannot influence those rules. This keeps them independent and prevents data leakage.
3. What does one-hot encoding accomplish?
One-hot encoding converts categorical values into numeric indicator columns that machine-learning models can process. For example, hotel types become separate binary features instead of being assigned arbitrary numbers that might incorrectly imply rank or distance.
4. Why use handle_unknown="ignore"?
It prevents the pipeline from failing when validation, test, or future production data contains a category that was not present during training. The unknown category is represented without creating an error, so the pipeline can process new reservations reliably without being refitted.
5. Why treat agent as categorical?
Although agent IDs contain numbers, they identify different booking agents and do not measure a continuous quantity. Agent 200 is not twice as large or important as agent 100. Treating it as categorical prevents the model from assuming a false numeric order or distance between agents.
6. Does a tree-based model require scaling?
Tree-based models generally do not require scaling because they split features using thresholds and are not affected by differences in measurement scale. I retained scaling in the reusable preprocessing pipeline so the same pipeline can support models that do require scaling, such as logistic regression, support-vector machines, and distance-based models. It also creates a consistent preprocessing process for future experiments.
7. How does a pipeline reduce leakage and deployment errors?
A scikit-learn pipeline groups preprocessing steps into one repeatable object. It ensures that imputation, scaling, encoding, feature order, and category handling are applied consistently during training, evaluation, and production. Because the pipeline is fitted only on training data and can be saved with the learned rules, it reduces both data leakage and discrepancies between development and deployment.

## Short Answers
1. Why did you use a chronological split?
I used a chronological split because the model will be trained on historical reservations and used to predict future cancellations. A random split could mix older and newer bookings across the datasets, creating an overly optimistic evaluation. The chronological split more realistically tests whether the model generalizes to future booking behavior.
2. Why use fit_transform() for training but transform() for validation and testing?
fit_transform() allows the preprocessing pipeline to learn rules—such as medians, means, scaling values, and category vocabularies—from the training data and then apply them. Validation and test data use only transform() so they cannot influence those rules. This keeps them independent and prevents data leakage.
3. What does one-hot encoding accomplish?
One-hot encoding converts categorical values into numeric indicator columns that machine-learning models can process. For example, hotel types become separate binary features instead of being assigned arbitrary numbers that might incorrectly imply rank or distance.
4. Why use handle_unknown="ignore"?
It prevents the pipeline from failing when validation, test, or future production data contains a category that was not present during training. The unknown category is represented without creating an error, so the pipeline can process new reservations reliably without being refitted.
5. Why treat agent as categorical?
Although agent IDs contain numbers, they identify different booking agents and do not measure a continuous quantity. Agent 200 is not twice as large or important as agent 100. Treating it as categorical prevents the model from assuming a false numeric order or distance between agents.
6. Does a tree-based model require scaling?
Tree-based models generally do not require scaling because they split features using thresholds and are not affected by differences in measurement scale. I retained scaling in the reusable preprocessing pipeline so the same pipeline can support models that do require scaling, such as logistic regression, support-vector machines, and distance-based models. It also creates a consistent preprocessing process for future experiments.
7. How does a pipeline reduce leakage and deployment errors?
A scikit-learn pipeline groups preprocessing steps into one repeatable object. It ensures that imputation, scaling, encoding, feature order, and category handling are applied consistently during training, evaluation, and production. Because the pipeline is fitted only on training data and can be saved with the learned rules, it reduces both data leakage and discrepancies between development and deployment.

## Milestone 3 completion checklist

- [ ] All TODO code cells are completed.
- [ ] All explanation prompts are answered in your own words.
- [ ] Training, validation, and test dates do not overlap.
- [ ] `reservation_status`, `reservation_status_date`, and other leakage fields are absent.
- [ ] The preprocessor is fitted only on training data.
- [ ] Train, validation, and test matrices have the same number of transformed columns.
- [ ] `models/preprocessing_pipeline.joblib` exists locally and remains ignored by Git.
- [ ] All three Milestone 3 report CSV files exist in `reports/`.
- [ ] The notebook and reports are committed and pushed to the Milestone 3 branch.
- [ ] You can answer the seven interview questions above without reading the notebook.
